# P44 — Aprendizaje residual profundo para reconocimiento de imágenes

## 1. Título y paper

**Paper:** *Deep Residual Learning for Image Recognition*  
**Autoría:** Kaiming He, Xiangyu Zhang, Shaoqing Ren, Jian Sun  
**Año y venue:** 2015 · arXiv:1512.03385 · CVPR 2016  
**Nivel:** L3 · **Motor:** `resnet`  
**Ficha completa:** [`P44_resnet`](../../papers/foundational/P44_resnet/README.md)

**Hito:** El atajo identidad hace apilables cientos de capas. Es la misma idea aditiva de la LSTM, aplicada a la profundidad.

- [arXiv:1512.03385](https://arxiv.org/abs/1512.03385)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Al pasar de 20 a 56 capas, el error de ENTRENAMIENTO subía. No era sobreajuste: era que las redes muy profundas se habían vuelto imposibles de optimizar.
2. Ejecutar una implementación mínima de la propuesta: Que cada bloque aprenda un residuo F(x) y la salida sea F(x) + x. Si la capa no aporta, aprender F ≈ 0 es fácil, y el gradiente siempre tiene una ruta directa.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P03
- P04
- P43


## 4. Intuición

Si una capa nueva no aporta, debería poder no hacer nada. Con capas normales, «no hacer nada» —la identidad— es sorprendentemente difícil de aprender. Con un atajo, es gratis: basta con que el bloque aprenda cero.


## 5. Concepto mínimo

```text
Bloque plano   :  y = F(x)          aprender la identidad es difícil
Bloque residual:  y = F(x) + x      la identidad es F ≡ 0

    ∂y/∂x = F'(x) + 1     ← el 1 sostiene el producto a través de las capas
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('resnet', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Qué gradiente queda tras 152 capas con un factor de 0,85 por capa?
2. ¿Y con el atajo?
3. ¿Por qué el paper llama «degradación» al problema y no «sobreajuste»?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('resnet', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('resnet', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Sin atajo, el gradiente a 152 capas es del orden de 1e-11: la señal no llega. Con atajo se mantiene en un rango utilizable. La diferencia es de nueve órdenes de magnitud, y explica por qué de golpe se pudieron entrenar redes diez veces más profundas.


## 10. Comentario pedagógico

La observación clave del paper es que el error de **entrenamiento** subía con la profundidad. Eso descarta el sobreajuste: no era falta de capacidad, era imposibilidad de optimizar. Distinguir ambas cosas es una habilidad diagnóstica que sirve para toda la vida.


## 11. Error o anti-patrón deliberado

Anti-patrón: diagnosticar «sobreajuste» sin mirar el error de entrenamiento.


In [ ]:
casos = [('entrenamiento bajo, validacion alta', 'sobreajuste'),
         ('entrenamiento ALTO, validacion alta', 'subajuste u optimizacion rota'),
         ('entrenamiento sube al anadir capas', 'DEGRADACION: el caso de ResNet')]
for sintoma, diagnostico in casos:
    print(f'{sintoma:<40} → {diagnostico}')

## 12. Corrección

El mismo principio aditivo aparece en tres sitios de este eje:


In [ ]:
principio = {'LSTM (P03)': 'c_t = f*c_{t-1} + i*g — ruta aditiva en el TIEMPO',
             'ResNet (P44)': 'y = F(x) + x — ruta aditiva en la PROFUNDIDAD',
             'Transformer (P08)': 'LayerNorm(x + Sublayer(x)) — en cada subcapa'}
show(principio)

## 13. Desafío guiado

Calcula a partir de cuántas capas el gradiente sin atajo baja de 1e-6 con factor 0,85.


In [ ]:
r = run_paper_lab('resnet', seed=3)['result']
show(r)

## 14. Desafío autónomo

Entrena dos redes de 30 capas, con y sin atajos, sobre un conjunto pequeño. Compara el error de **entrenamiento**: si el plano es peor, has reproducido la degradación.


## 15. Evidencia de aprendizaje

Guarda la tabla de gradientes por profundidad, la tabla de diagnóstico y tu explicación de por qué la identidad es difícil sin atajo.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P44_resnet/README.md) · evaluación formal: [`assessments/papers/P44_resnet.md`](../../assessments/papers/P44_resnet.md)


## 16. Cierre

Ya se entrenan redes enormes. El problema pasa a ser el contrario: cómo servirlas sin arruinarse.


## 17. Conexión con el siguiente hito

- P08
- toda arquitectura profunda posterior

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
